In [1]:
import pandas as pd

In [2]:
day_0 = pd.read_csv('day_0.csv', parse_dates = ['day_0'])
user_journal = pd.read_csv('user_journal.csv', parse_dates = ['active_day'])

In [3]:
print(day_0.shape)
print(day_0.dtypes)
display(day_0.head())
print(user_journal.shape)
print(user_journal.dtypes)
display(user_journal.head())

(4319, 2)
user_pseudo_id            object
day_0             datetime64[ns]
dtype: object


,user_pseudo_id,day_0
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15
2,6B41795D5E5B7339E007330941C9E201,2018-06-15
3,0FB42D5FFB79A9DE147873753BF7A664,2018-07-15
4,61C77C8D9E7F92524289DA7E9D4786BF,2018-07-15


(59037, 2)
user_pseudo_id            object
active_day        datetime64[ns]
dtype: object


,user_pseudo_id,active_day
0,102CC7D8CD5D3E861EACE1BAB97E61EC,2018-06-26
1,D5AA8A02352703280A47B61A01621DD2,2018-06-26
2,1F25763EC8BCAEC7FB069E81659A2153,2018-06-26
3,CAB801809B633E8717EB78894D440EBF,2018-06-26
4,2741C47CC50AE98B39CC5B561BFEA504,2018-06-26


In [4]:
day_0_x_active_day = pd.merge(day_0, user_journal, on = 'user_pseudo_id', how = 'inner')
print(day_0_x_active_day.shape)
day_0_x_active_day

(11346, 3)


,user_pseudo_id,day_0,active_day
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15,2018-06-15
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15,2018-06-16
2,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15,2018-06-15
3,6B41795D5E5B7339E007330941C9E201,2018-06-15,2018-06-17
4,6B41795D5E5B7339E007330941C9E201,2018-06-15,2018-06-15
...,...,...,...
11341,383F4B98EC31DD2A915427D36A30354E,2018-08-04,2018-08-08
11342,383F4B98EC31DD2A915427D36A30354E,2018-08-04,2018-08-13
11343,CCADFC0ECC097AA5DA4FD3917D517022,2018-08-04,2018-08-04
11344,4542CFA974FAA3C923527BDDA50397B0,2018-08-04,2018-08-04


In [5]:
day_0_x_active_day['days_between'] = (day_0_x_active_day['active_day'] - day_0_x_active_day['day_0']).dt.days
print(day_0_x_active_day.shape)
display(day_0_x_active_day.head())
print(day_0_x_active_day['days_between'].min())
print(day_0_x_active_day['days_between'].max())

(11346, 4)


,user_pseudo_id,day_0,active_day,days_between
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15,2018-06-15,0
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15,2018-06-16,1
2,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15,2018-06-15,0
3,6B41795D5E5B7339E007330941C9E201,2018-06-15,2018-06-17,2
4,6B41795D5E5B7339E007330941C9E201,2018-06-15,2018-06-15,0


-96
111


In [6]:
contaminated = day_0_x_active_day[day_0_x_active_day['days_between'] < 0]
print(contaminated.shape)
print(contaminated['user_pseudo_id'].nunique())


(130, 4)
12


In [54]:
day_0_x_active_day['retained_d1'] = day_0_x_active_day['days_between'] == 1
day_0_x_active_day['retained_d3'] = day_0_x_active_day['days_between'] == 3
day_0_x_active_day['retained_d7'] = day_0_x_active_day['days_between'] == 7
day_0_x_active_day['retained_d14'] = day_0_x_active_day['days_between'] == 14
day_0_x_active_day['retained_d30'] = day_0_x_active_day['days_between'] == 30
player_flags = day_0_x_active_day.groupby('user_pseudo_id', as_index = False).agg({'day_0': 'first', 'retained_d1': 'any', 'retained_d3': 'any', 'retained_d7': 'any', 'retained_d14': 'any', 'retained_d30': 'any'})
display(player_flags.head())
print(player_flags.shape)

,user_pseudo_id,day_0,retained_d1,retained_d3,retained_d7,retained_d14,retained_d30
0,000A8B9167F5F9B7BF620708D320E32F,2018-07-25,False,False,False,False,False
1,0018E9BAF92AFA2FFFCDCB5F48A3B134,2018-10-02,False,False,False,False,False
2,00353D3418DD0707FE0A80A59FBD5705,2018-09-11,True,True,False,False,False
3,0043CF8841C819015ACD25710D333113,2018-07-26,True,False,False,False,False
4,005F127942F3FD25588272950FDFD339,2018-08-15,False,False,False,False,False


(4319, 7)


In [90]:
cutoffs = {
    'd1': (1, '2018-10-02'),
    'd3': (3, '2018-09-30'),
    'd7': (7, '2018-09-26'),
    'd14': (14, '2018-09-19'),
    'd30': (30, '2018-09-03')
}

rows = []
for day, (day_num, date) in cutoffs.items():
    eligible = player_flags[player_flags['day_0'] <= date]

    n_eligible = eligible.shape[0]
    n_retained = eligible[f'retained_{day}'].sum()
    pct = round(100 * n_retained / n_eligible, 2) if n_eligible > 0 else 0

    rows.append({
        'N': day,
        'N_days': day_num,
        'eligible': n_eligible,
        'retained': n_retained,
        'rate_pct': pct
    })

retention_long = pd.DataFrame(rows)
retention_long

,N,N_days,eligible,retained,rate_pct
0,d1,1,4251,915,21.52
1,d3,3,4151,435,10.48
2,d7,7,3998,223,5.58
3,d14,14,3711,137,3.69
4,d30,30,2963,62,2.09
